# 04 - EDA: systematic near-duplicate / label-noise scan

Follow-up to the `presence`/`prsence` typo spotted manually in `notebooks/01_eda_text_overview.ipynb` (image `zF6Y0Q8CW0z7x0EU` clearly shows "presence" handwritten but is labeled `prsence`). This notebook scans **all** train lines for near-duplicate pairs (fuzzy match, not just exact) to find whether that's an isolated typo or part of a broader pattern, and to separate genuine label noise from expected repeated legal boilerplate with real period spelling/abbreviation variation across different source documents.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process

from src.data import load_raw_data

train = load_raw_data('train')
texts = train['Target'].tolist()
ids = train['ID'].tolist()

# Full pairwise similarity matrix - fast with rapidfuzz's C implementation
# (~1s for ~4k short strings), infeasible with pure-Python difflib at this scale.
scores = process.cdist(texts, texts, scorer=fuzz.ratio, dtype=np.uint8)
scores.shape

(4098, 4098)

In [2]:
def cluster_near_duplicates(scores, min_ratio, max_ratio=100):
    """Union-find clusters of lines whose pairwise ratio falls in [min_ratio, max_ratio)."""
    n = scores.shape[0]
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    pairs = []
    for i in range(n):
        row = scores[i]
        for j in range(i + 1, n):
            s = row[j]
            if min_ratio <= s < max_ratio:
                pairs.append((i, j, int(s)))
                union(i, j)

    from collections import defaultdict
    clusters = defaultdict(list)
    involved = {i for i, j, s in pairs} | {j for i, j, s in pairs}
    for idx in involved:
        clusters[find(idx)].append(idx)
    return pairs, clusters


broad_pairs, broad_clusters = cluster_near_duplicates(scores, min_ratio=88)
len(broad_pairs), len(broad_clusters)

(423, 50)

## Most near-duplicate clusters are genuine repeated boilerplate, not errors

This is a corpus of 17th-century legal deeds: standard opening/closing formulas ("To all Christian people to whome these presents shall come...", "By this publique Act and Instrument of protest...") recur across many different original documents. Below is one such cluster - the variation (`Christian`/`christian`/`Xpian`, `presents`/`present`/`pnts`, `known`/`knowne`) reflects real scribal spelling and abbreviation differences between different documents/scribes, not transcription mistakes. `Xpian` (chi-rho abbreviation for "Christian") and `pnts`/`p^rsent` (abbreviation marks) are genuine period conventions that the ground truth correctly preserves - normalizing them would erase real signal, not clean noise.

In [3]:
example_root = max(broad_clusters, key=lambda r: len(broad_clusters[r]))
for idx in broad_clusters[example_root]:
    print(ids[idx], '|', texts[idx])

TtFzl8vV3RmMxcXg | To all Christian People to whom this present wrighting
kXCBCqNUYbythYj6 | To all Xpian People to whome this presents shall
H6MlB7cGEGI4lZ6J | To all Christian People to whome this present writeing shall come
ewYlEIEmas0JBluR | To all Christian People to whome this present writeing shall come
0ZvGzksVcRwCqqvb | To all Christian People to whome these presents shall
gg5hfS9d6SHxZS3e | To all christian people to whome these presents shall
tlIG4kS1IsC4cvjc | To all Christian People to whome these presents shall
7G2Eyw9CxUwwivGA | To all Christian people to whome these presents Shall
RAalP94Vy9npUxzr | To all Christian people to whome these presents shall
ChtGx4491s5eJoNw | To all Christian people to whome this prsent writing
Jrm3V9E0DcaSAlZF | To all Christian people to whom this present writing
7S3dq4Ychd8teqvS | To all Christian people to whome these presents writeinge shall come knowe
HPvWOlCY08BG2lbK | To all xpian People to whome this p^rsent wr
KJd5sqATAxGLuiV4 | To

## Isolating the `presence` / `prsence` case

Narrowing to a tight similarity threshold (ratio >= 95, i.e. differing by only 1-3 characters) surfaces the same phrase repeated with a single-letter difference - the pattern the manual check in notebook 01 flagged.

In [4]:
tight_pairs, tight_clusters = cluster_near_duplicates(scores, min_ratio=95)

presence_root = next(
    r for r, members in tight_clusters.items()
    if any('presence' in texts[m].lower() or 'prsence' in texts[m].lower() for m in members)
)
presence_members = tight_clusters[presence_root]

n_prsence = sum('prsence' in texts[m].lower() for m in presence_members)
n_presence = sum('presence' in texts[m].lower() for m in presence_members)
len(presence_members), n_prsence, n_presence

(5, 3, 2)

Unlike the `Xpian`/`pnts`-style clusters above, `prsence` carries no abbreviation mark (no `^`, no recognized period shorthand). An earlier version of notebook 01 claimed the handwriting for `zF6Y0Q8CW0z7x0EU` "unambiguously" reads "presence" in full - on closer inspection that was an overclaim; the cursive ligature between `p` and `s` is compressed and genuinely hard to disambiguate. Across the whole train set the split is close to even (19 `prsence` vs 17 `presence` occurrences, see notebook 01), which at least rules out a rare one-off typo, but that alone doesn't tell us which spelling (if either) is "correct" - it's consistent with either a real transcriber inconsistency or a genuinely ambiguous, consistently-written ligature that different transcribers read differently in good faith.

## Decision

**Do not normalize or "correct" `Target` spelling anywhere in the pipeline** (features, model post-processing, or training data). Reasons:
1. The vast majority of near-duplicate lines (this notebook's broad    scan) reflect genuine document-to-document variation in a    formulaic legal corpus, not noise - "fixing" them would erase    real period spelling/abbreviations the ground truth correctly    preserves.
2. Even for the confirmed `presence`/`prsence` inconsistency, `src/   evaluate.py::combined_score` scores against whatever text Zindi    provides. We have no visibility into the hidden test labels, so    there's no way to know whether "correcting" spelling would help    or hurt CER/WER there - and if the same transcriber inconsistency    exists in test labels too, correcting predictions to standard    spelling would actively hurt the score.
3. Net effect: treat this as a small, irreducible noise floor in the    achievable CER/WER (not 100% attributable to model quality), and    keep it in mind during later error analysis so near-duplicate    mismatches like this aren't mistaken for genuine model errors.

Saving the full near-duplicate cluster table for reference during later error analysis (not used by any src/ code).

In [5]:
from pathlib import Path

rows = []
for cluster_id, members in broad_clusters.items():
    for m in members:
        rows.append({'cluster_id': cluster_id, 'ID': ids[m], 'Target': texts[m]})

near_dup_df = pd.DataFrame(rows).sort_values(['cluster_id', 'ID'])
out_path = Path('../data/processed/near_duplicate_clusters.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
near_dup_df.to_csv(out_path, index=False)
len(near_dup_df), out_path

(207, WindowsPath('../data/processed/near_duplicate_clusters.csv'))

## Update: reverted the correction - the visual evidence doesn't hold up

A prior version of this notebook reviewed all 19 `prsence`-labeled source images and concluded, based on the claim that `zF6Y0Q8CW0z7x0EU` "unambiguously" shows "presence" written in full, that this was a confirmed transcriber typo. That specific claim didn't survive a second look (see notebook 01's cell 7): the handwriting is genuinely ambiguous, not clearly one spelling or the other. Since the correction was justified entirely by that visual-certainty claim, and the claim was wrong, the correction isn't justified either.

**Reverted `src/data.py::TARGET_CORRECTIONS` and its test.** Back to the original decision in the Decision section above: do not normalize/correct `Target` spelling anywhere in the pipeline. Treat `presence`/`prsence` as a documented, unresolved ambiguity rather than a confirmed error - same reasoning as the broader near-duplicate boilerplate clusters, which were never touched.